# Setup & Data — deepTechno


## 1. Install dependencies


In [ ]:
!pip install -q pretty-midi mido timm
# torch and tensorflow are pre-installed on Colab T4

## 2. Clone & install the deepTechno package


In [ ]:
!git clone https://github.com/juan-garassino/AUD-deep-techno /content/repo
!pip install -q -e /content/repo

## 3. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/deep-techno-data'
import os; os.makedirs(DRIVE, exist_ok=True)

## 4. Download MAESTRO MIDI from Google Cloud Storage (~6 sec)


In [ ]:
import tensorflow as tf, zipfile, os
MIDI_DIR = '/content/maestro'
zip_path = tf.keras.utils.get_file(
    'maestro-v2.0.0-midi.zip',
    origin='https://storage.googleapis.com/magentadata/datasets/maestro/v2.0.0/maestro-v2.0.0-midi.zip',
    cache_dir='/content', cache_subdir='maestro_zip')
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(MIDI_DIR)
print('Extracted to', MIDI_DIR)

## 5. Regenerate split CSVs (path-agnostic, works in any environment)


In [ ]:
from deepTechno.data.sourcing import find_midi_files, split_csv
midi_files, csv_path = find_midi_files(MIDI_DIR, 'my-maestro.csv')
split_csv('my-maestro.csv', 'train.csv', 'val.csv', 'test.csv', data_dir=MIDI_DIR)
print(f'Found {len(midi_files)} MIDI files')
TRAIN_CSV = os.path.join(MIDI_DIR, 'train.csv')
VAL_CSV   = os.path.join(MIDI_DIR, 'val.csv')

## 6. Get all_notes.csv — choose ONE option below


### Option A — Upload directly from this machine (no Drive needed)
File gone when session ends.


In [ ]:
# Option A: upload from local machine
from google.colab import files
print('Select all_notes.csv from your local machine...')
uploaded = files.upload()
ALL_NOTES = '/content/all_notes.csv'
print('Uploaded:', list(uploaded.keys()))

### Option B — Read from Google Drive (recommended, persists across sessions)
Upload all_notes.csv to MyDrive/deep-techno-data/ once.


In [ ]:
# Option B: read directly from Drive mount (no copy needed)
ALL_NOTES = f'{DRIVE}/all_notes.csv'
if os.path.exists(ALL_NOTES):
    print('all_notes.csv found on Drive:', ALL_NOTES)
else:
    print('Not found. Upload all_notes.csv to MyDrive/deep-techno-data/ or use Option A/C.')

### Option C — Re-parse from MAESTRO (~15 min, no upload needed)


In [ ]:
# Option C: re-parse from MIDI (slow but no upload required)
import pandas as pd
from deepTechno.data.preprocess import midi_to_notes
import glob
ALL_NOTES = '/content/all_notes.csv'
if not os.path.exists(ALL_NOTES):
    midi_paths = glob.glob(f'{MIDI_DIR}/**/*.midi', recursive=True)
    print(f'Parsing {len(midi_paths)} MIDI files...')
    all_notes = pd.concat([midi_to_notes(p) for p in midi_paths], ignore_index=True)
    all_notes.to_csv(ALL_NOTES, index=False)
    import shutil; shutil.copy(ALL_NOTES, f'{DRIVE}/all_notes.csv')
    print(f'Parsed {len(all_notes):,} notes, saved to Drive')

## 7. Sanity check


In [ ]:
import pandas as pd
df = pd.read_csv(ALL_NOTES, nrows=5)
print(df)
print(f'\nTotal rows: {sum(1 for _ in open(ALL_NOTES)):,}')